# Delegation and Supervision

**Prerequisites:** `01_single_vs_multi_agent.ipynb`. From `03_tools`: `01_what_is_a_tool.ipynb`

Notebook 1 ran five specialists by hand: each was given its piece, and their pieces were collected. That works when you already know who does what. Most of the time you don't. A message arrives, and somebody has to decide who should deal with it.

A chapter 2 parent cannot make that decision, because it has no list of agents to choose from. A **supervisor** can. It holds a fixed roster of agents, reads each message, asks the one that should handle it, and deals with what comes back.

The recall from notebook 1 is now under way, and messages keep arriving: customers, journalists, the regulator, reports from the field.

**By the end you'll have:**
- Turned an agent into an ordinary chapter 3 tool, and seen that what a supervisor knows about an agent is its tool description, not its persona
- Watched a supervisor route five different messages, and seen what a careless description does to that routing
- Looked at what comes back from a specialist, and what the supervisor does with it
- Seen a supervisor decide who to ask next from what the last agent found
- Compared delegating with handing a conversation over, on the same customer chat

## Setup

`supervisor.py` now holds `Agent` (moved from notebook 1), plus what this notebook adds. `recall.py` holds the recall: the brief, the specialists' personas, and a small database behind their tools. `meter.py` is notebook 1's `Meter`.

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd().parent
sys.path.append(str(HERE))
for chapter in ("01_foundations", "02_agent_runtime", "03_tools"):
    sys.path.append(str(HERE.parent / chapter))

import json

import recall
from chat import Conversation
from meter import Meter
from registry import Registry
from supervisor import MODEL, Agent, Chat, Think, as_tool, supervisor

## The roster

Four specialists from notebook 1, each with the brief, and now with real tools where their job needs one. Support can look up a serial number. The engineer can find which battery cell lot a unit was built with, and what testing found for that lot. The database is small and made up, but what it returns is fixed, so every run is about the same facts.

Each agent now has two pieces of text. Its **persona** is its system prompt: how to *be* this agent. Its **description** is what other agents are told about it: when to *ask* it. They are written for different readers.

In [2]:
def specialist(name, tools=()):
    return Agent(name, f"{recall.PERSONAS[name]}\n\n{recall.BRIEF}", recall.DESCRIPTIONS[name],
                 tools=Registry(list(tools)))

support = specialist("support", [recall.lookup_serial])
pr = specialist("pr")
lawyer = specialist("lawyer")
engineer = specialist("engineer", [recall.cell_lot, recall.test_results])
roster = [support, pr, lawyer, engineer]

for agent in roster:
    print(f"{agent.name:<9} tools: {', '.join(agent.tools.names) or '-':<25} {agent.description}")

print()
print(recall.lookup_serial(serial="AR2-2409-01544"))
print(recall.cell_lot(serial="AR2-2409-01544"))

support   tools: lookup_serial             Customer support. Writes replies to customers who own the product: whether their unit is affected, what to do, how to get a refund or replacement. Can look up any serial number.
pr        tools: -                         PR. Writes statements and replies for journalists and the public.
lawyer    tools: -                         Product lawyer. Writes to the product-safety regulator, including updates to the recall notification.
engineer  tools: cell_lot, test_results    Hardware engineering. Assesses technical reports and field failures: whether a unit could have the battery fault, and whether the fault reaches beyond the recalled batch. Can look up a unit's cell lot and the test results for any lot.

AR2-2409-01544: batch AR2-2409, shipped 2-16 September. not in the recall.
AR2-2409-01544 was built with battery cells from lot VC-7731.


That last pair is the catch the database is built around. Unit AR2‑2409‑01544 is from September's batch, which is *not* in the recall, but it was built with cells from the same lot as the recalled batch. Support's lookup cannot see that. The engineer's can. Section 4 depends on it.

## An agent is a tool

A supervisor needs a way to ask another agent something and get an answer back. Chapter 3 already has exactly that shape: a `Tool` is a function with a description, and the model decides when to call it. So an agent becomes a tool with one argument:

```python
def as_tool(agent):
    def ask(request: Annotated[str, "Everything this specialist needs to know to do the job. "
                                    "They see nothing else: not the original message, not other replies."]) -> str:
        return agent.run(request)
    return Tool(ask, name=f"ask_{agent.name}", description=agent.description)
```

Here is everything a supervisor's model is told about the engineer:

In [3]:
print(json.dumps(as_tool(engineer).schema, indent=1))

{
 "type": "function",
 "function": {
  "name": "ask_engineer",
  "description": "Hardware engineering. Assesses technical reports and field failures: whether a unit could have the battery fault, and whether the fault reaches beyond the recalled batch. Can look up a unit's cell lot and the test results for any lot.",
  "parameters": {
   "additionalProperties": false,
   "properties": {
    "request": {
     "description": "Everything this specialist needs to know to do the job. They see nothing else: not the original message, not other replies.",
     "type": "string"
    }
   },
   "required": [
    "request"
   ],
   "type": "object"
  }
 }
}


Three things follow from this.

- **The description is the only thing routing can go on.** The engineer's persona, its bluntness, its tools, the brief: none of it is in the schema. A supervisor decides who to ask from two sentences somebody wrote about the agent.
- **The request is the whole of what the agent will know.** It is chapter 2's self-contained goal again, and the parameter's description says so, because the supervisor writes that request on the spot.
- **Nothing new was needed.** The supervisor below is an ordinary `Agent` whose tools happen to be other agents. It runs chapter 2's loop unchanged: `Think` picks a tool, `Act` calls it, which runs a whole other agent, and `Observe` puts the reply back.

In [4]:
SUPERVISOR = """You run the recall desk for the Aria 2 battery recall. Every incoming message comes to you first, and you never write replies yourself.

Decide which specialist should handle each message and ask them, giving them everything they need to know: they see nothing but what you send them. If what one specialist finds means others need to act, ask them too. When you are done, give the reply to send back, or, if several people had to act, say briefly who did what."""

log = []
desk = supervisor(SUPERVISOR, roster, log=log)
print(desk.tools)

<Registry 4 tools: ask_support, ask_pr, ask_lawyer, ask_engineer>


## Choosing who to ask

Five messages of the kind a recall desk gets. For each, only the supervisor's *first* decision is taken: which specialist it asks, and what it asks them. Nothing is run beyond that, so this costs one call per message.

In [5]:
MESSAGES = {
    "customer":   "From a customer: Hi, my headphones have serial AR2-2408-01422. Are they part of the recall, and what should I do?",
    "journalist": "From a journalist at TechDaily: Has anyone been hurt, and are any other batches affected?",
    "regulator":  "From the product-safety regulator: Please confirm the serial range covered by the recall and the date it began.",
    "lab":        "From the QA lab: A cross-section of 5 cells from lot VC-7731 shows separators 12 microns thick, against a 16 micron specification.",
    "social":     "A public post on social media, tagging the company: 'my aria 2 smells like burning plastic when it charges?? is this the recall thing'",
}

def first_choice(agent, message):
    conversation = Conversation(system=agent.persona)
    conversation.messages.append({"role": "user", "content": message})
    decision = Think(model=MODEL, tools=agent.tools.tools, allow_fork=False, explain=False).run(conversation)
    if decision.get("type") != "tool_call":
        return "(answered itself)", decision["content"]
    return decision["tool"], json.loads(decision["content"]).get("request", "")

def route(agent):
    for kind, message in MESSAGES.items():
        tool, request = first_choice(agent, message)
        print(f"{kind:<11} -> {tool:<14} {request}\n")

with Meter() as meter:
    route(desk)
print(meter.summary())

customer    -> ask_support    Check serial AR2-2408-01422 for recall eligibility and provide next steps.



journalist  -> ask_engineer   Check if any other batches are affected by the Aria 2 battery recall. Also confirm if any injuries have been reported in connection with the recall.



regulator   -> ask_lawyer     Please confirm the serial range covered by the recall and the date it began.



lab         -> ask_engineer   Given that the QA lab has found that a cross-section of 5 cells from lot VC-7731 shows separators that are 12 microns thick, against a 16 micron specification, please assess the risk: are these units likely to exhibit the battery fault? Should this lot be included in the recall?



social      -> ask_engineer   Assess whether a unit of the Aria 2 could have the battery fault that causes burning plastic smell during charging, and whether the fault could affect units beyond the recalled batch. Provide any relevant test results and cell lot information.

{'calls': 5, 'prompt': 2986, 'completion': 1524, 'total': 4510, 'seconds': 4}


Three of the five go where you would expect: the customer to support, the regulator to the lawyer, the lab measurement to engineering.

**The journalist went to engineering.** PR's description says "journalists and the public", and it was not chosen. The supervisor routed by *what the question needs looking up* rather than by *who is asking*. The social post went to engineering too, in this run. In an earlier run of the same cell it went to PR, which is the first hint that one run per message does not settle anything. Later in this notebook the same question gets measured properly.

Look at the requests as well as the choices. Each one restates the message in the supervisor's own words, because the specialist will see nothing else. When that restatement drops something, the specialist works from less than the supervisor had. There is an example of that below.

### A careless description

Descriptions get written once and then forgotten. Here is a plausible one for PR: *"Communications. Writes anything that goes outside the company about the recall."* It is not wrong. PR does write for outside the company. But so do support, who write to customers, and the lawyer, who writes to the regulator.

In [6]:
careless_pr = Agent("pr", pr.persona, "Communications. Writes anything that goes outside the company about the recall.")
careless_desk = supervisor(SUPERVISOR, [support, careless_pr, lawyer, engineer])

with Meter() as meter:
    route(careless_desk)
print(meter.summary())

customer    -> ask_support    Check if serial AR2-2408-01422 is part of recall and provide next steps.



journalist  -> ask_engineer   Assess whether the battery fault could have affected other batches beyond the recalled ones. Provide any findings regarding additional batches and whether any injuries have been reported.



regulator   -> ask_lawyer     Please confirm the serial range covered by the recall and the date it began.



lab         -> ask_engineer   Assess whether a cross-section of 5 cells from lot VC-7731 shows separators 12 microns thick (vs 16 micron spec) is relevant to the Aria 2 battery recall. Determine if this defect could cause the recall fault or if it extends beyond the recalled batch. Provide recommendations for next steps.



social      -> ask_engineer   Does the burning plastic smell during charging indicate that the battery is affected by the Aria 2 recall?

{'calls': 5, 'prompt': 2991, 'completion': 900, 'total': 3891, 'seconds': 4}


The routing came out the same as before: support, engineering, the lawyer, engineering, engineering. On this evidence the careless description changed nothing.

That is a weak statement, and it should be. One run per message cannot show a small change in routing, in either direction. To say anything, the same message has to be run many times and the choices counted, which is what the next-but-one section does for a different change. It costs a call per trial, so it is worth doing for the routing you depend on, not for every description.

## What comes back

Now one message all the way through: the journalist's. The supervisor asks a specialist, the specialist does its work, and a reply comes back to the supervisor, which then gives its own final answer. The log shows what the specialist actually wrote, which is otherwise buried in the supervisor's conversation.

In [7]:
log.clear()
with Meter() as meter:
    final = desk.run(MESSAGES["journalist"])
costs = {"journalist": meter.summary()}

for entry in log:
    print(f"=== asked {entry['agent']}:\n{entry['request']}\n\n=== {entry['agent']} replied:\n{entry['reply']}\n")
print(f"=== the supervisor's final answer:\n{final}\n")
print(costs["journalist"])

=== asked engineer:
Check if any other batches beyond the recalled batch are affected by the battery fault.

=== engineer replied:
400 units from other batches were stress‑tested. None showed the fault.

=== asked support:
Provide information about any reported injuries related to the Aria 2 battery recall and confirm if any other batches beyond the recalled batch are affected.

=== support replied:
Two customers reported minor burns – one to a finger and one to an ear – when the headphones got hot while charging. No fires happened, and neither person needed a hospital visit.

The fault has only been found in the batch with serial numbers AR2‑2408‑00001 to AR2‑2408‑03200 (shipped 5‑19 August). We tested 400 units from other batches and none showed the problem. So, no other batches are affected.

=== the supervisor's final answer:
Yes, there have been a few reports of injuries. Two customers sustained minor burns—one on a finger and one on an ear—when their headphones overheated while c

Two things are worth pulling out of that.

**PR was never asked.** The supervisor asked engineering, then support, and wrote the reply to the journalist itself. Its own prompt says it never writes replies, and what it produced is close to a paraphrase of support's answer. So the reply that would go to a journalist was written in the supervisor's voice by way of a customer-support answer, which is exactly the mixing notebook 1 spent its length on.

**A reply is a plain string.** Support wrote *"no other batches are affected"*, and the supervisor passed it on. The database disagrees: the first 1,800 units of batch AR2‑2409 were built from the same cell lot as the recalled batch. Support cannot see that; only engineering's lookup can. Nothing in what came back could say *"this is the part I am sure about"*, *"this came from the brief, not from a tool"*, or *"do not send this outside the company"*. A string has nowhere to put any of it, and the supervisor has nothing to check. That gap is what notebook 3 is about.

## Deciding who's next from what came back

Two customer emails that look almost the same. Both say a unit got hot while charging. Neither unit is from the recalled batch. The supervisor gets each one and decides everything after that itself.

In [6]:
REPORTS = {
    "A": "From a customer, by email: 'My Aria 2 got really hot while charging last night and the case has bulged a bit. "
         "Serial AR2-2409-01544. I'm fine, just worried.'",
    "B": "From a customer, by email: 'My Aria 2 got warm while charging from my laptop's 65 W charger. No swelling or "
         "anything like that. Serial AR2-2407-00210. Is that normal?'",
}

def trace(report, agent=None, label=None):
    log.clear()
    with Meter() as meter:
        final = (agent or desk).run(REPORTS[report])
    costs[label or f"report {report}"] = meter.summary()
    for step, entry in enumerate(log, 1):
        print(f"--- {step}. asked {entry['agent']}: {entry['request']}\n\n{entry['reply']}\n")
    print(f"=== the supervisor's final answer:\n{final}\n")
    key = label or f"report {report}"
    print(key, costs[key])

In [9]:
trace("A")

--- 1. asked support: Customer reports: Serial AR2-2409-01544. Unit got hot while charging, bulged case. They are fine but worried. Need to determine if this unit is part of recall and provide next steps.

Your serial number, AR2‑2409‑01544, is not in the batch that is being recalled. That batch is AR2‑2408‑00001 to AR2‑2408‑03200 (shipped 5‑19 August).  

So your headphones aren’t part of the recall. You can keep using them on battery power. If you ever see the battery swell or the headphones get hot while charging again, stop using it, unplug the charger, and let us know. For now, you’re safe.

=== the supervisor's final answer:
Your serial number, AR2‑2409‑01544, is not in the batch that is being recalled. That batch is AR2‑2408‑00001 to AR2‑2408‑03200 (shipped 5‑19 August).  

So your headphones aren’t part of the recall. You can keep using them on battery power. If you ever see the battery swell or the headphones get hot while charging again, stop using it, unplug the charger, and

In [10]:
trace("B")

--- 1. asked support: Customer reports that Aria 2 unit (serial AR2-2407-00210) got warm while charging from a 65 W laptop charger. No swelling or other issues observed. They are asking whether this is normal behavior. Please advise if this unit is affected by the recall and what steps to take.

**Stop charging the headphones right now. Let them cool off and watch them.**

Your headphone is serial **AR2‑2407‑00210**.  
It came from batch **AR2‑2407**, shipped 8‑22 July. That batch is **not part of the recall** (the recall covers only batch AR2‑2408, shipped 5‑19 August).

In the recall batch, 14 customers reported the headphones getting hot while charging, and 2 people suffered minor burns. The problem appears only when the headphones are charging, and it is related to a battery separator defect in that specific batch.

For your unit:  
* If it feels warm but does not swell, smells, or becomes scorching, it is likely normal.  
* If it ever starts to swell, catch fire, or you feel a bur

Both reports went to support and stopped there.

For report A this is the dangerous one. The customer says the case has **bulged**, which is the same symptom as the recall. Support looked the serial up, saw it was not on the recall list, and answered:

> "So your headphones aren't part of the recall. You can keep using them on battery power… **For now, you're safe.**"

The unit is from batch AR2‑2409, which is not recalled, but `cell_lot` says it was built from lot VC‑7731 — the lot behind the recall. Support has no way to see that; `lookup_serial` only answers "is this serial on the list". Engineering has the lookup that would have caught it, and engineering was never asked.

Report B went the same way, and support's answer contradicts itself: it opens with "Stop charging your headphones right now" and then says warmth is probably normal.

The supervisor routed by **who sent the message**. A customer email goes to customer support. Nothing in what it was given made the content of the message — bulging, overheating — count for more than the sender.

### Saying what an agent cannot do

A description says when to ask an agent. None of these say when *not* to, and that is the gap report A fell into. Support's description promises "whether their unit is affected", which sounds like the whole question. It is really "whether this serial is on the recall list", which is a much smaller thing, and the difference is invisible from the outside.

So each description gets one more sentence, saying where the agent's knowledge stops and who to ask instead. Nothing else changes: same agents, same personas, same tools, same supervisor prompt.

In [7]:
SHARPER = {
    "support": recall.DESCRIPTIONS["support"] +
               " Only knows whether a serial is on the recall list; cannot tell whether a unit outside the recall "
               "could have the fault. Ask engineering about that first.",
    "pr": recall.DESCRIPTIONS["pr"] +
          " Ask before replying to anyone outside the company about how far the problem reaches.",
    "lawyer": recall.DESCRIPTIONS["lawyer"] +
              " Must be told whenever a new finding changes what the regulator was told.",
    "engineer": recall.DESCRIPTIONS["engineer"] +
                " Ask first about any unit that overheated, swelled or burned, whatever batch it is from, and before "
                "anyone tells a customer their unit is safe.",
}

sharper_roster = [Agent(a.name, a.persona, SHARPER[a.name], tools=a.tools) for a in roster]
sharper_desk = supervisor(SUPERVISOR, sharper_roster, log=log)

for agent in sharper_roster:
    print(f"{agent.name}: {agent.description}\n")

support: Customer support. Writes replies to customers who own the product: whether their unit is affected, what to do, how to get a refund or replacement. Can look up any serial number. Only knows whether a serial is on the recall list; cannot tell whether a unit outside the recall could have the fault. Ask engineering about that first.

pr: PR. Writes statements and replies for journalists and the public. Ask before replying to anyone outside the company about how far the problem reaches.

lawyer: Product lawyer. Writes to the product-safety regulator, including updates to the recall notification. Must be told whenever a new finding changes what the regulator was told.

engineer: Hardware engineering. Assesses technical reports and field failures: whether a unit could have the battery fault, and whether the fault reaches beyond the recalled batch. Can look up a unit's cell lot and the test results for any lot. Ask first about any unit that overheated, swelled or burned, whatever batc

In [8]:
trace("A", sharper_desk, "report A (sharper)")

--- 1. asked support: Check if serial AR2-2409-01544 is on the recall list and advise the customer accordingly. Also, if it is on recall, what steps to take.

Your serial number AR2‑2409‑01544 is **not part of the recall**. It belongs to batch AR2‑2409, shipped on 2‑16 September.  

You can keep using the headphones normally. If you ever feel them get hot while charging, stop charging and let them cool. No recall replacement or refund is needed for this unit.

--- 2. asked engineer: Unit AR2-2409-01544 overheated while charging, case bulged. Serial AR2-2409-01544. Determine if this unit could have the battery fault, check cell lot and test results. Provide recommendation for the customer. Also determine if this fault could affect other units beyond recall list.

- **Cell lot:** VC‑7731  
- **Test results for lot VC‑7731:** 11 returned units opened → 9 show a separator defect that lets the cell overheat while charging; 2 cause not yet determined.  
- **Serial AR2‑2409‑01544** comes from

In [9]:
trace("B", sharper_desk, "report B (sharper)")

--- 1. asked support: Check recall status for serial AR2-2407-00210.

AR2-2407-00210 is not in the recall.

--- 2. asked engineer: Assess whether overheating during charging with a 65W charger is a known defect for this serial. Check cell lot and test results.

I need the specific serial number to look up its cell lot and the test results for that lot.

--- 3. asked engineer: Check if overheating during charging with a 65W charger is a known defect for the serial AR2-2407-00210. Provide cell lot and test results.

Cell lot: VC‑7702  
Test results: 150 units stress‑tested at 45 °C while charging – no overheating, no swelling, separators within spec.

=== the supervisor's final answer:
Stopping: reached the maximum of 8 iterations. Last step: decision -> '{"request":"Check if overheating during charging with a 65W charger is a known defect for the serial AR2-2407-00210. Provide cell lot and test results."}'. This is partial progress, not a final answer.

report B (sharper) {'calls': 9, '

With the extra sentences, this run did reach engineering, though not first. Support was still asked first and still answered *"not part of the recall… You can keep using the headphones normally"*, and only then did the supervisor ask engineering, which found lot VC‑7731 and the separator defect. The final answer to the customer is the right one: stop charging, return it, replacement or refund.

Report B shows two other things, both worth having.

- **The request is the whole of what a specialist sees.** The supervisor asked engineering *"Assess whether overheating during charging with a 65 W charger is a known defect for this serial"* — without the serial. Engineering answered *"I need the specific serial number to look up its cell lot."* A whole round trip was spent on a missing word.
- **A supervisor's budget is spent in whole agents.** After that wasted round trip the run hit `max_iterations=8` and ended with *"Stopping: reached the maximum of 8 iterations… This is partial progress, not a final answer."* The customer would have got nothing. Chapter 2's budgets count iterations of the supervisor's own loop, and each of those iterations can be an entire other agent's run. Two things about that limit are worth knowing: `Runtime` counts every *component* it runs, and this loop has three, so 8 was under three turns — much tighter than it looks. `Agent`'s default was raised to 24 while notebook 3 was being written; this run is left as it happened.

So: better than before on report A, and nothing conclusive yet. Whether the descriptions caused the improvement or the run was lucky is not something one run can answer.

### The same rule, one level up

The descriptions did not move it. The other place a routing rule can live is the supervisor's own instructions, which it reads on every message, rather than in a tool schema it reads while choosing. Same rule, same agents, same sharper descriptions; only the supervisor's prompt changes.

In [8]:
RULE = ("\n\nAny report of a unit overheating, swelling or burning goes to engineering first, whatever batch it is "
        "from, and before anyone tells a customer their unit is safe.")

ruled_desk = supervisor(SUPERVISOR + RULE, sharper_roster, log=log)
print(ruled_desk.persona)

You run the recall desk for the Aria 2 battery recall. Every incoming message comes to you first, and you never write replies yourself.

Decide which specialist should handle each message and ask them, giving them everything they need to know: they see nothing but what you send them. If what one specialist finds means others need to act, ask them too. When you are done, give the reply to send back, or, if several people had to act, say briefly who did what.

Any report of a unit overheating, swelling or burning goes to engineering first, whatever batch it is from, and before anyone tells a customer their unit is safe.


In [11]:
trace("A", ruled_desk, "report A (rule)")

--- 1. asked engineer: Serial number AR2-2409-01544: overheating while charging, bulging case. Need to determine if unit could have battery fault and whether it belongs to recalled batch. Also check cell lot and test results.

- **Cell lot**: VC‑7731 (same lot used in recalled batch).  
- **Recall status**: Serial AR2‑2409‑01544 is **not** in the recalled batch (AR2‑2408‑00001 to AR2‑2408‑03200).  
- **Test results for lot VC‑7731**: 11 returned units from this lot – 9 had a separator defect that lets the cell overheat while charging; 2 had undetermined causes.  
- **Conclusion**: The unit could have the same battery fault (separator defect) that caused overheating in the recalled batch, but it is not part of the current recall.

--- 2. asked support: Customer: Serial AR2-2409-01544. Engineering: not in recall but could have separator defect. Customer concerned about overheating and bulging case. Need to advise safe steps, possible replacement or further investigation. Provide info for

In [12]:
trace("B", ruled_desk, "report B (rule)")

--- 1. asked engineer: Customer reports that their Aria 2 (serial AR2-2407-00210) got warm while charging from a 65 W laptop charger. No swelling or any other visible issue. The customer asks if this is normal. Please assess whether this unit could have the battery fault and if this overheating could be due to the recalled fault or other cause. Include any relevant info about the batch or cell lot if available.

- **Batch & cell lot**: Serial AR2‑2407‑00210 was built with cells from lot **VC‑7702**.  
- **Recall relevance**: The recalled batch (AR2‑2408‑00001 – AR2‑2408‑03200) used cells from lot VC‑7731. This unit is *not* from that batch.  
- **Fault evidence**: No data in the brief indicates a separator defect in lot VC‑7702. The only confirmed fault is in VC‑7731.  
- **Overheating**: The warm sensation during charging could be normal for a 65 W charger, but without testing of VC‑7702 we cannot confirm or rule out a defect.  
- **Action**: The unit is outside the current recall. No

With the rule in the supervisor's prompt, report A went to engineering **first**: cell lot VC‑7731, the separator defect, not in the recalled batch but the same fault is possible. The supervisor then asked support to write the customer's reply, which opens *"Stop charging the headphones right now"* and explains why. That is the answer the first version should have given.

Report B went to engineering first as well, took four calls, and came back with the right answer the other way: lot VC‑7702, tested clean, *"It's not a sign of the recall defect"*, with advice to watch it. The rule costs an extra agent call on messages that do not need one, which is what buying a guarantee looks like.

### How often, not whether

Every trace above is one run, and one run of a model is a coin flip rather than a result. The question worth answering is how *often* each version asks engineering first about report A, so each is run eight times. Only the supervisor's first decision is taken, which is the decision in question and costs one call.

In [10]:
TRIALS = 8
VERSIONS = {"plain": desk, "sharper": sharper_desk, "rule": ruled_desk}

routes = {}
with Meter() as meter:
    for label, agent in VERSIONS.items():
        routes[label] = [first_choice(agent, REPORTS["A"])[0] for _ in range(TRIALS)]
costs["routing trials"] = meter.summary()

for label, picks in routes.items():
    asked = sum(p == "ask_engineer" for p in picks)
    print(f"{label:<8} engineer first {asked}/{TRIALS}   " + ", ".join(p.replace("ask_", "") for p in picks))
print()
print(costs["routing trials"])

plain    engineer first 1/8   support, engineer, support, support, support, support, support, support
sharper  engineer first 0/8   support, support, support, support, support, support, support, support
rule     engineer first 8/8   engineer, engineer, engineer, engineer, engineer, engineer, engineer, engineer

{'calls': 24, 'prompt': 16456, 'completion': 6214, 'total': 22670, 'seconds': 92}


In [11]:
from math import comb

def fisher(x1, n1, x2, n2):
    """One-sided Fisher exact test: how surprising is x1/n1 if both versions behaved the same."""
    successes = x1 + x2
    return sum(comb(n1, k) * comb(n2, successes - k)
               for k in range(x1, min(n1, successes) + 1)) / comb(n1 + n2, successes)

counts = {label: sum(p == "ask_engineer" for p in picks) for label, picks in routes.items()}
for better, worse in (("sharper", "plain"), ("rule", "plain"), ("rule", "sharper")):
    print(f"{better} ({counts[better]}/{TRIALS}) vs {worse} ({counts[worse]}/{TRIALS}): "
          f"p = {fisher(counts[better], TRIALS, counts[worse], TRIALS):.3f}")

sharper (0/8) vs plain (1/8): p = 1.000
rule (8/8) vs plain (1/8): p = 0.001
rule (8/8) vs sharper (0/8): p = 0.000


| | engineer asked first |
|---|---|
| plain descriptions | 1 of 8 |
| sharper descriptions | 0 of 8 |
| the rule in the supervisor's prompt | **8 of 8** |

Sharper descriptions against plain: p = 1.0, which is no effect at all. The rule against either: p ≤ 0.001.

**The two sentences added to the descriptions did nothing to the first choice**, and the earlier trace that looked like an improvement was a coin landing the other way. This is why the traces above are labelled as examples: run each of them again and they can come out differently.

Be precise about what was measured. This counts the **first** specialist asked. The sharper run did eventually reach engineering, on its second step, so a description can still change what happens later in a run. What it did not do is change who gets asked first, which is the decision that let report A be answered by support alone.

Why the difference. The supervisor's prompt is read on every message, as part of who the supervisor is. A tool description is read while choosing between tools, and on a model this size it carries much less weight. **A rule that must always hold belongs in the supervisor's instructions; a description says what an agent is for, and cannot be relied on to carry a rule.**

**This is measured on one model.** Everything here runs on gpt-oss-20b, and how much weight a model gives a tool description against its own system prompt is very likely a matter of size and training. A larger model may read the descriptions properly and need no rule at all. What carries across is the method rather than the number: if routing matters, put the rule where your model actually reads it, and count the choices rather than assuming.

The 24 trials cost 22,670 tokens, about the same as two full traced runs. Measuring routing is not free, but it costs less than trusting it.

## Handing over instead of asking

Everything so far is **delegation**: the supervisor asks, gets an answer, and stays in charge. Every message comes to it first, and every reply goes back through it.

A **handoff** moves the conversation instead. The agent that hands over steps out, and the one it hands to talks to the person directly from then on, seeing everything said so far. `Chat` in `supervisor.py` does this with one more tool per target, `transfer_to_<name>`. Calling it stops the current agent's loop through the `cancel` hook chapter 2 uses to stop abandoned branches, and the new owner answers the same turn.

The same three-message customer chat, both ways. First a front desk that hands over:

In [11]:
FRONT_DESK = """You are the first point of contact on the Aria 2 recall line. You never answer questions yourself. Work out who the person is and what they need, and hand the conversation to the right colleague."""

CUSTOMER_CHAT = [
    "Hi, I think my headphones might be part of the recall?",
    "The serial is AR2-2408-01422.",
    "OK. I actually got a small burn on my finger from them last week. Do I need to do anything about that?",
]

front_desk = Agent("front_desk", FRONT_DESK, "The recall line's front desk.")
handed = Chat(front_desk, handoffs={"front_desk": [support, pr]})

with Meter() as meter:
    for message in CUSTOMER_CHAT:
        reply = handed.send(message)
        print(f"customer: {message}\n{handed.transcript[-1]['agent']}: {reply}\n")
costs["handoff chat"] = meter.summary()
print("transfers:", handed.transfers)
print(costs["handoff chat"])

customer: Hi, I think my headphones might be part of the recall?
support: Can you share your headphone’s serial number? It’ll be something like “AR2‑2408‑01422.” Once I have it, I can let you know if it’s in the recall batch and what to do next.



customer: The serial is AR2-2408-01422.
support: Your headphones are part of the recall.  
**What you need to do right now**

1. **Stop charging** the headphones immediately.  
2. Go to [aria.example.com/recall](https://aria.example.com/recall).  
3. Choose a free replacement or a full refund.  
4. The site will send you a prepaid return package. Pack the headphones in that box and mail them back.

Once we receive them, we’ll ship you a new pair or issue your refund.  

Sorry for the hassle—let me know if you need help with the website or the return.



customer: OK. I actually got a small burn on my finger from them last week. Do I need to do anything about that?
support: I’m really sorry you were burned.  

Because the burn happened while the headphones were charging, it’s part of the recalled batch. Please:

1. **Stop using the headphones while charging** (or any charging at all) right now.  
2. **Seek medical attention** if you still have pain, redness, or swelling. A doctor can check the burn for you.  
3. Let us know by visiting the recall page again—[aria.example.com/recall](https://aria.example.com/recall)—and choose the replacement or refund option.  
4. The prepaid return package we send will also have a small form for you to fill out. Tell us about the burn when you do that; it helps us track the incident.

We’re sorry for what happened and are here to help you through this. If you need anything else, just let me know.

transfers: [{'from': 'front_desk', 'to': 'support', 'reason': 'Customer is inquiring about a possible rec

And a front desk that delegates. It keeps the conversation, asks support whenever it needs to, and passes the answer on:

In [12]:
FRONT_DESK_DELEGATING = """You are the first point of contact on the Aria 2 recall line, talking with a member of the public. You never answer questions yourself: for anything they need, ask the right colleague, giving them everything they need to know since they see nothing else, and pass their answer on."""

log.clear()
delegating = supervisor(FRONT_DESK_DELEGATING, [support, pr], log=log)
transcript = []

with Meter() as meter:
    for message in CUSTOMER_CHAT:
        transcript.append({"role": "user", "content": message})
        reply = delegating.respond(transcript)
        transcript.append({"role": "assistant", "content": reply})
        print(f"customer: {message}\nfront desk: {reply}\n")
costs["delegated chat"] = meter.summary()
print(costs["delegated chat"])

customer: Hi, I think my headphones might be part of the recall?
front desk: **Stop charging the headphones right away.**  
If you’re still plugging them in, unplug them and leave them on a flat, cool surface.  
The risk only happens while the headphones are charging.

**How to find out if yours is affected**

1. Look on the back of the headphones for the serial number (it starts with “AR2‑”).  
2. Compare it to the recalled batch numbers: **AR2‑2408‑00001 – AR2‑2408‑03200**.  
3. If your number falls in that range, your headphones are part of the recall.

**What to do next**

- Go to **aria.example.com/recall**.  
- Enter your serial number and you’ll see the option for a free replacement or a full refund.  
- We’ll mail you prepaid return packaging. Just drop the headphones in the box, seal it, and mail it back.  
- Once we receive it, you’ll get a replacement or your money back.

**If you’re not in the recall batch**

- Keep using the headphones as you normally would.  
- They’re sa

customer: The serial is AR2-2408-01422.
front desk: **Your headphones are in the recall.**

1. **Stop charging immediately.**  
   – Do not plug them into a charger.  
   – If you’re already charging, unplug them right away and let them sit on a flat, non‑flammable surface.  
   – Use them only while they’re powered by the battery.

2. **Choose your next step** – free replacement or full refund.  
   – Go to **aria.example.com/recall**.  
   – Pick the option that suits you.  
   – The recall portal will give you a prepaid return label and instructions.

3. **When the prepaid return package arrives**  
   – Place the headphones inside the box.  
   – Seal the box as instructed.  
   – Drop it off at the specified drop‑off location.

**If your serial number had not been in the recall, you could keep using the headphones safely as long as you never charge them.**



customer: OK. I actually got a small burn on my finger from them last week. Do I need to do anything about that?
front desk: Stop charging the headphones right away – don’t plug them in again.

1. **Get medical help** – If you haven’t already, see a doctor or visit an urgent‑care clinic for the burn. Tell them the burn happened with the headphones while charging.

2. **Report the recall** – Go to aria.example.com/recall.  
   * Enter your serial number (AR2‑2408‑01422).  
   * Choose whether you want a free replacement or a full refund.  
   * The site will send you prepaid return packaging. Pack the headphones carefully and return them.

3. **Keep the charger and headphones** – Don’t throw them away. We need them to understand the issue.

{'calls': 11, 'prompt': 9180, 'completion': 2998, 'total': 12178, 'seconds': 34}


What support was given to work from, each way. With the handoff, support saw the conversation itself. With delegation, it saw only what the front desk wrote in each request:

In [13]:
print("=== handoff: support's view is the whole transcript, including the front desk's turns")
for m in handed.transcript:
    print(f"  {m['role']:<9} {m.get('agent', ''):<11} {m['content'][:110]}")

print("\n=== delegation: support's view, one request at a time")
for n, entry in enumerate(e for e in log if e["agent"] == "support"):
    print(f"  request {n + 1}: {entry['request']}")

=== handoff: support's view is the whole transcript, including the front desk's turns
  user                  Hi, I think my headphones might be part of the recall?
  assistant support     Can you share your headphone’s serial number? It’ll be something like “AR2‑2408‑01422.” Once I have it, I can 
  user                  The serial is AR2-2408-01422.
  assistant support     Your headphones are part of the recall.  
**What you need to do right now**

1. **Stop charging** the headphon
  user                  OK. I actually got a small burn on my finger from them last week. Do I need to do anything about that?
  assistant support     I’m really sorry you were burned.  

Because the burn happened while the headphones were charging, it’s part o

=== delegation: support's view, one request at a time
  request 1: Hi, I think my headphones might be part of the recall? I need to confirm if they are affected, what to do next, and how to get a refund or replacement. Please provide any details an

**The handoff.** The front desk transferred on the very first message, with a note for support. From then on support talked to the customer directly: it asked for the serial, looked it up, and answered in its own voice. Five calls, 4,891 tokens.

**The delegation.** The front desk stayed in charge for all three turns, asked support each time, and rewrote every answer. Eleven calls, 12,178 tokens: **2.5×** the handoff for the same conversation.

It also drifted. Its first reply, before anyone had a serial number, told the customer:

> "**If you're not in the recall batch** — keep using the headphones as you normally would. They're safe on battery; just don't charge them."

Nobody said that. "Don't charge them" is advice for a recalled unit, applied to units that are not recalled. The more a front desk rewrites, the more chances it has to invent, and it rewrites on every turn.

**What each side saw.** After the handoff, support had the whole transcript, the customer's words included. With delegation, support saw three separate requests, each one the front desk's summary. The third one did carry the serial number forward, because the front desk chose to write it in; nothing made it.

**What each is good for.** A handoff suits a conversation that belongs to one specialist from the moment you know who it is: cheaper, fewer hands, the specialist's own voice. Delegation suits work where the requests keep changing who they need, because the supervisor is still there to send the next one somewhere else. After a handoff, the front desk is out of the conversation, and the new owner needs its own handoffs to pass it on again.

## What it cost

In [13]:
print(f"{'':<16}{'calls':>6}{'prompt':>9}{'completion':>12}{'total':>8}{'seconds':>9}")
for name, c in costs.items():
    print(f"{name:<16}{c['calls']:>6}{c['prompt']:>9}{c['completion']:>12}{c['total']:>8}{c['seconds']:>9}")

                 calls   prompt  completion   total  seconds
journalist           5     3698        1158    4856        6
report A             3     2264         898    3162        3
report B             4     3373        1562    4935        4
report A (sharper)     8     7210        2911   10121       18
report B (sharper)     9     7547        1982    9529       21
report A (rule)      8     7519        2153    9672       20
report B (rule)      4     3646        1755    5401        4
routing trials      24    16456        6214   22670       92
handoff chat         5     4258         633    4891        4
delegated chat      11     9180        2998   12178       34


- **Routing itself is cheap.** One decision is about 600 to 1,200 tokens. Everything expensive is the work that follows it, so the thing worth getting right is which agent the work goes to.
- **Each extra specialist is a whole agent run.** Report B under the rule is four calls; report A under it is eight. Notebook 1's fork paid for branches that were all the same agent; here you pay for the one you ask, plus a supervisor call each time round.
- **Delegation costs more than a handoff per turn of a conversation**, 12,178 against 4,891 here, because the supervisor pays to read the conversation and to rewrite the answer every turn.
- **The measurement cost 22,670 tokens**, more than any single trace in this notebook. It is also the only number here that supports a claim.

## Where this leaves us

A supervisor is an ordinary agent whose tools are other agents. That is all the machinery needed, and chapter 2's loop runs it unchanged. What the notebook found beyond that:

- **A supervisor knows an agent by its description**, not by its persona, its tools or what it can see. Write the description for the reader that has to choose.
- **The request is the whole of what the chosen agent gets.** Leave the serial number out and the specialist asks for it, at the price of another agent run.
- **Routing you depend on belongs in the supervisor's instructions.** Measured over eight trials each: a sharpened description changed the first choice 0 of 8 times, a rule in the prompt 8 of 8.
- **Delegation keeps the supervisor in charge; a handoff gives the conversation away.** The handoff was 2.5× cheaper here and kept the specialist's own voice; delegation kept the ability to send the next turn elsewhere.
- **Results come back as plain strings.** Nothing in a reply can say "I am not sure", "this is internal", or "ask engineering about this part", so the supervisor cannot check any of it. Notebook 3 gives replies a shape.

Two failures are worth carrying forward to notebook 5, where failure handling is the subject: a run that ended on *"reached the maximum of 8 iterations"* with nothing for the customer, and a supervisor that answered a journalist with a claim the database contradicts, because it never asked the agent that could have checked.